# Análise de operações financeiras - Nível 1
## Parte A - Análise determinística com Pandas

In [1]:
import json
from pathlib import Path
import pandas as pd

In [ ]:
caminho_dados = Path("../dados/dados_nivel_1.json")
Path.cwd()
caminho_dados.exists()

True

In [20]:

with open(caminho_dados, "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

taxa_cambio_usd_brl = dados["taxa_cambio_usd_brl"]

df = pd.DataFrame(dados["operacoes"])
df


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [21]:
df.isna().sum()


id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [22]:
df["id"].duplicated().sum()

np.int64(1)

In [23]:
df["moeda"].value_counts(dropna=False)
df["canal"].value_counts(dropna=False)
df["tipo"].value_counts(dropna=False)

tipo
transferencia_enviada     11
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

In [ ]:
df["cliente_id"].nunique()

# 20 operacoes, e 6 clientes

6

In [29]:
df[df["id"].duplicated(keep=False)].sort_values("id")

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [40]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["data"].dtype

dtype('<M8[us]')

In [47]:
df = df.drop_duplicates().copy()

df["data"] = pd.to_datetime(df["data"], errors="coerce")

df["data_ausente"] = df["data"].isna()

In [48]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False


In [49]:
print("Quantidade de registros:", len(df))
print("Duplicidades integrais:", df.duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Clientes únicos:", df["cliente_id"].nunique())

Quantidade de registros: 19
Duplicidades integrais: 0
Datas ausentes: 1
Clientes únicos: 6


In [50]:
colunas_categoricas = ["canal", "tipo"]

for coluna in colunas_categoricas:
    df[coluna] = df[coluna].str.strip().str.lower()

In [51]:
for coluna in colunas_categoricas:
    print(f"\nValores de {coluna}:")
    print(df[coluna].value_counts(dropna=False))


Valores de canal:
canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

Valores de tipo:
tipo
transferencia_enviada     10
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64


In [52]:
df["moeda"] = df["moeda"].str.strip().str.upper()

In [53]:
df["moeda"].value_counts(dropna=False)

moeda
BRL    18
USD     1
Name: count, dtype: int64

In [54]:
df["valor"].describe()

count       19.000000
mean     11194.736842
std       8085.272873
min       1400.000000
25%       4050.000000
50%       8800.000000
75%      17250.000000
max      27000.000000
Name: valor, dtype: float64

In [55]:
df["valor_brl"] = df["valor"]

In [ ]:
df["valor_brl"] = df["valor"].astype(float)

# Identificar operações em USD
mascara_usd = df["moeda"] == "USD"

# Converter USD para BRL
df.loc[mascara_usd, "valor_brl"] = (
    df.loc[mascara_usd, "valor"].astype(float)
    * float(taxa_cambio_usd_brl)
)

In [61]:
df
# versão final do dataframe após as normalizações

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,3300.0
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False,25900.0
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False,27000.0
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,17200.0
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,15200.0
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False,16100.0
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False,3800.0


In [62]:
print("Quantidade de registros:", len(df))
print("Clientes únicos:", df["cliente_id"].nunique())
print("Duplicidades integrais:", df.duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Valores BRL ausentes:", df["valor_brl"].isna().sum())
print("Moedas encontradas:", df["moeda"].unique())

Quantidade de registros: 19
Clientes únicos: 6
Duplicidades integrais: 0
Datas ausentes: 1
Valores BRL ausentes: 0
Moedas encontradas: <StringArray>
['BRL', 'USD']
Length: 2, dtype: str


In [63]:
volume_por_cliente = (
    df.groupby("cliente_id", as_index=False)
      .agg(volume_total_brl=("valor_brl", "sum"))
      .sort_values("volume_total_brl", ascending=False)
)

In [64]:
volume_por_cliente


,cliente_id,volume_total_brl
3,CLI-A-4,79500.0
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


In [65]:
quantidade_por_canal = (
    df.groupby("canal", as_index=False)
      .agg(quantidade_operacoes=("id", "count"))
      .sort_values("quantidade_operacoes", ascending=False)
)

In [66]:
quantidade_por_canal

,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


In [67]:
total_base = df["valor_brl"].sum()
total_agregado = volume_por_cliente["volume_total_brl"].sum()

print("Total da base:", total_base)
print("Total agregado:", total_agregado)
print("Totais iguais:", total_base == total_agregado)

Total da base: 265500.0
Total agregado: 265500.0
Totais iguais: True


In [68]:
print(
    "Contagem por canal igual ao total de registros:",
    quantidade_por_canal["quantidade_operacoes"].sum() == len(df)
)

Contagem por canal igual ao total de registros: True


In [69]:
df_regra_1 = df[df["data"].notna()].copy()

In [70]:
resumo_regra_1 = (
    df_regra_1
    .groupby(["cliente_id", "data"], as_index=False)
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_operacoes_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
)

In [71]:
resumo_regra_1["sinalizado_regra_1"] = (
    (resumo_regra_1["quantidade_operacoes"] >= 3)
    & (resumo_regra_1["soma_operacoes_brl"] > 50_000)
    & (resumo_regra_1["maior_operacao_brl"] < 20_000)
)

In [72]:
casos_regra_1 = resumo_regra_1[
    resumo_regra_1["sinalizado_regra_1"]
].copy()

casos_regra_1

,cliente_id,data,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True


In [73]:
operacoes_sinalizadas_regra_1 = df_regra_1.merge(
    casos_regra_1[["cliente_id", "data"]],
    on=["cliente_id", "data"],
    how="inner"
)

operacoes_sinalizadas_regra_1[
    [
        "id",
        "cliente_id",
        "data",
        "valor",
        "moeda",
        "valor_brl",
        "canal",
        "tipo",
        "contraparte"
    ]
]

,id,cliente_id,data,valor,moeda,valor_brl,canal,tipo,contraparte
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,18100.0,pix,transferencia_enviada,Alfa Comercio LTDA
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,17300.0,pix,transferencia_enviada,Alfa Comercio LTDA
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,18800.0,ted,transferencia_enviada,Beta Servicos ME


In [74]:
validacao_positiva = resumo_regra_1[
    (resumo_regra_1["cliente_id"] == "CLI-A-1")
    & (resumo_regra_1["data"] == pd.Timestamp("2026-03-09"))
]

validacao_positiva

,cliente_id,data,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True


In [75]:
assert len(validacao_positiva) == 1
assert validacao_positiva.iloc[0]["quantidade_operacoes"] == 3
assert validacao_positiva.iloc[0]["soma_operacoes_brl"] == 54_200
assert validacao_positiva.iloc[0]["maior_operacao_brl"] == 18_800
assert bool(validacao_positiva.iloc[0]["sinalizado_regra_1"]) is True

print("Caso positivo validado corretamente.")

Caso positivo validado corretamente.


In [76]:
validacao_negativa = resumo_regra_1[
    (resumo_regra_1["cliente_id"] == "CLI-A-3")
    & (resumo_regra_1["data"] == pd.Timestamp("2026-03-05"))
]

validacao_negativa

,cliente_id,data,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False


In [77]:
assert len(validacao_negativa) == 1
assert validacao_negativa.iloc[0]["quantidade_operacoes"] == 3
assert validacao_negativa.iloc[0]["soma_operacoes_brl"] == 48_500
assert validacao_negativa.iloc[0]["maior_operacao_brl"] == 17_200
assert bool(validacao_negativa.iloc[0]["sinalizado_regra_1"]) is False

print("Caso negativo validado corretamente.")

Caso negativo validado corretamente.


In [103]:
import math

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["valor_brl"],
    64_800.0,
    rel_tol=1e-9,
    abs_tol=0.01
)

In [104]:
estatisticas_clientes = (
    df.groupby("cliente_id", as_index=False)
      .agg(
          quantidade_operacoes=("id", "count"),
          mediana_cliente_brl=("valor_brl", "median")
      )
)

estatisticas_clientes

,cliente_id,quantidade_operacoes,mediana_cliente_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


In [105]:
analise_regra_2 = df.merge(
    estatisticas_clientes,
    on="cliente_id",
    how="left"
)

In [106]:
analise_regra_2[
    [
        "id",
        "cliente_id",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_cliente_brl"
    ]
]

,id,cliente_id,valor_brl,quantidade_operacoes,mediana_cliente_brl
0,OP-0001,CLI-A-1,18100.0,4,17700.0
1,OP-0002,CLI-A-1,17300.0,4,17700.0
2,OP-0003,CLI-A-1,18800.0,4,17700.0
3,OP-0004,CLI-A-1,3300.0,4,17700.0
4,OP-0005,CLI-A-2,25900.0,2,26450.0
5,OP-0006,CLI-A-2,27000.0,2,26450.0
6,OP-0007,CLI-A-3,17200.0,3,16100.0
7,OP-0008,CLI-A-3,15200.0,3,16100.0
8,OP-0009,CLI-A-3,16100.0,3,16100.0
9,OP-0010,CLI-A-4,3800.0,4,5450.0


In [107]:
analise_regra_2["limite_5x_mediana_brl"] = (
    analise_regra_2["mediana_cliente_brl"] * 5
)

In [108]:
analise_regra_2["sinalizado_regra_2"] = (
    (analise_regra_2["quantidade_operacoes"] >= 4)
    & (
        analise_regra_2["valor_brl"]
        > analise_regra_2["limite_5x_mediana_brl"]
    )
)

In [109]:
casos_regra_2 = analise_regra_2[
    analise_regra_2["sinalizado_regra_2"]
].copy()

casos_regra_2[
    [
        "id",
        "cliente_id",
        "data",
        "valor",
        "moeda",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_cliente_brl",
        "limite_5x_mediana_brl",
        "sinalizado_regra_2"
    ]
]

,id,cliente_id,data,valor,moeda,valor_brl,quantidade_operacoes,mediana_cliente_brl,limite_5x_mediana_brl,sinalizado_regra_2
12,OP-0013,CLI-A-4,2026-03-24,12000,USD,64800.0,4,5450.0,27250.0,True


In [110]:
validacao_positiva_regra_2 = analise_regra_2[
    analise_regra_2["id"] == "OP-0013"
]

validacao_positiva_regra_2[
    [
        "id",
        "cliente_id",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_cliente_brl",
        "limite_5x_mediana_brl",
        "sinalizado_regra_2"
    ]
]

,id,cliente_id,valor_brl,quantidade_operacoes,mediana_cliente_brl,limite_5x_mediana_brl,sinalizado_regra_2
12,OP-0013,CLI-A-4,64800.0,4,5450.0,27250.0,True


In [112]:
import math

assert len(validacao_positiva_regra_2) == 1
assert validacao_positiva_regra_2.iloc[0]["quantidade_operacoes"] == 4

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["mediana_cliente_brl"],
    5_450.0,
    abs_tol=0.01
)

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["limite_5x_mediana_brl"],
    27_250.0,
    abs_tol=0.01
)

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["valor_brl"],
    64_800.0,
    abs_tol=0.01
)

assert bool(
    validacao_positiva_regra_2.iloc[0]["sinalizado_regra_2"]
) is True

print("Caso positivo da Regra 2 validado corretamente.")

Caso positivo da Regra 2 validado corretamente.


In [113]:
clientes_nao_elegiveis = analise_regra_2[
    analise_regra_2["quantidade_operacoes"] < 4
]

assert not clientes_nao_elegiveis["sinalizado_regra_2"].any()

print("Clientes com menos de quatro operações não foram sinalizados.")

Clientes com menos de quatro operações não foram sinalizados.
